In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


In [3]:
# 1. Load Dataset
df = pd.read_csv("Housing.csv")

In [4]:
df.head()
df.shape
df.isnull().sum()

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

In [5]:
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [6]:

# 2. Preprocessing
target = "price"

numeric_features = [
    "area", "bedrooms", "bathrooms", "stories", "parking"
]

categorical_features = [
    "mainroad", "guestroom", "basement", "hotwaterheating",
    "airconditioning", "prefarea", "furnishingstatus"
]

In [8]:
X = df[numeric_features + categorical_features]
y = df[target]

In [9]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [10]:
# 3. Linear Regression Pipeline
linear_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

linear_pipeline = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", LinearRegression())
])

In [11]:
linear_pipeline.fit(X_train, y_train)
linear_pred = linear_pipeline.predict(X_test)

linear_r2 = r2_score(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_mae = mean_absolute_error(y_test, linear_pred)


In [12]:
# 4. Polynomial Regression Pipeline
poly_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

poly_pipeline = Pipeline([
    ("preprocessor", poly_preprocessor),
    ("model", LinearRegression())
])

poly_pipeline.fit(X_train, y_train)
poly_pred = poly_pipeline.predict(X_test)

poly_r2 = r2_score(y_test, poly_pred)
poly_rmse = np.sqrt(mean_squared_error(y_test, poly_pred))
poly_mae = mean_absolute_error(y_test, poly_pred)

In [14]:
# 5. Compare Performance
print(f"Linear Regression    R2: {linear_r2:.4f}, RMSE: {linear_rmse:,.2f}, MAE: {linear_mae:,.2f}")
print(f"Polynomial Regression R2: {poly_r2:.4f}, RMSE: {poly_rmse:,.2f}, MAE: {poly_mae:,.2f}")


Linear Regression    R2: 0.6529, RMSE: 1,324,506.96, MAE: 970,043.40
Polynomial Regression R2: 0.6496, RMSE: 1,330,810.07, MAE: 978,314.39


In [15]:
# Based on this dataset's supplied data, Linear Regression performs slightly better.
best_model = linear_pipeline if linear_r2 >= poly_r2 else poly_pipeline
print("\nSelected Model:", "Linear Regression" if best_model is linear_pipeline else "Polynomial Regression")


Selected Model: Linear Regression


In [16]:
# 6. Predictive Function
def predict_house_price(
    area, bedrooms, bathrooms, stories, parking,
    mainroad, guestroom, basement, hotwaterheating,
    airconditioning, prefarea, furnishingstatus
):
    new_house = pd.DataFrame([{
        "area": area,
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "stories": stories,
        "parking": parking,
        "mainroad": mainroad,
        "guestroom": guestroom,
        "basement": basement,
        "hotwaterheating": hotwaterheating,
        "airconditioning": airconditioning,
        "prefarea": prefarea,
        "furnishingstatus": furnishingstatus
    }])

    prediction = best_model.predict(new_house)[0]
    return prediction

In [17]:
predicted_price = predict_house_price(
    area=7000,
    bedrooms=3,
    bathrooms=2,
    stories=2,
    parking=1,
    mainroad="yes",
    guestroom="no",
    basement="yes",
    hotwaterheating="no",
    airconditioning="yes",
    prefarea="yes",
    furnishingstatus="semi-furnished"
)

print(f"\nPredicted House Price: {predicted_price:,.2f}")


Predicted House Price: 7,423,441.38
